# 23. AI2D — LLM-reasoner поверх распарсенной диаграммы (не VLM)

Парадигма **CoG-DQA** (Wang et al., CVPR 2024, 0.79 на AI2D) и MM-CoT: диаграмма **парсится
в текст**, а ответ выбирает **текстовый LLM рассуждением** — без зрения (не VLM).

### Зачем
Граф/ColBERT-подходы на AI2D упираются в ~0.35 (текстовый матчинг исчерпан). Тот же
распарсенный материал, поданный LLM-reasoner'у, даёт **~0.65** (наш замер на gpt-oss-20b),
почти вдвое выше — потому что LLM **рассуждает** (+ фоновые знания), а не сопоставляет векторы.

### Конвейер
1. **Парсинг диаграммы → текст** (DPTs у нас уже есть): `caption` (short_description) +
   OCR-подписи `ocr_v2` с грубыми позициями (top/middle/bottom × left/center/right).
2. **MCQ-промпт:** сериализация + вопрос + варианты A/B/C/D.
3. **LLM-reasoner** (OpenAI-совместимый роутер из `.env`): выбирает букву.
4. **Метрика:** accuracy на test, сравнение моделей.

### Известное ограничение
Вопросы «что обозначает буква D/E» требуют привязки буква→регион (визуальной), которой в
текстовой сериализации нет → часть таких ответов теряется. Закрывается добавлением регионов
(OpenCV/SAM2) в сериализацию — вне этого ноутбука.

In [ ]:
import os, sys, json, re, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from openai import OpenAI

p = Path.cwd()
while p != p.parent:
    if (p / 'src' / 'vqa_retrieval').exists() and (p / 'notebooks').exists():
        break
    p = p.parent
ROOT = p; DATA_ROOT = ROOT.parent
load_dotenv(ROOT / '.env')

MANIFEST   = DATA_ROOT / 'ai2d' / 'prepared_v2' / 'manifest_hybrid.jsonl'
OUTPUT_DIR = ROOT / 'runs' / 'ai2d_llm_reasoner'; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT, '| key set:', bool(os.environ.get('OPENAI_API_KEY')),
      '| base_url:', os.environ.get('OPENAI_BASE_URL'))

ROOT: c:\Users\Jet\Desktop\data\diagram_vqa | key set: True | base_url: https://routerai.ru/api/v1


In [ ]:
MODELS  = ['openai/gpt-oss-20b', 'deepseek/deepseek-v4-pro'] 
N_EVAL  = None 
WORKERS = 8               
MAX_TOK = 1536           
OCR_MIN_CONF, MAX_LINES = 35.0, 30
LETTERS = 'ABCDEFGH'

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'], base_url=os.environ.get('OPENAI_BASE_URL'))
print('models:', MODELS, '| N_EVAL:', 'ALL test' if N_EVAL is None else N_EVAL, '| workers:', WORKERS)

models: ['openai/gpt-oss-20b', 'deepseek/deepseek-v4-pro'] | N_EVAL: ALL test | workers: 8


---
## Парсинг диаграммы в текст

`caption` + OCR-строки с грубой позицией из bbox (нормируем по `image_size`). Это и есть
«distilled» представление диаграммы, которое читает LLM вместо пикселей.

In [ ]:
#  3. Загрузка test + сериализация диаграммы 
def load_test(n):
    rows = []
    with open(MANIFEST, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d.get('split') != 'test':
                continue
            opts = [str(o).strip() for o in d.get('options', []) if str(o).strip()]
            if len(opts) < 2:
                continue
            rows.append({'question': d['question'], 'options': opts,
                         'correct': max(0, min(int(d.get('correct_option_idx', 0)), len(opts) - 1)),
                         'caption': d.get('short_description', ''),
                         'ocr_path': str(DATA_ROOT / d['ocr_v2_path'])})
    random.Random(42).shuffle(rows)
    return rows if n is None else rows[:n]          # None = весь test

def coarse_pos(b, W, H):
    cx = (b[0] + b[2]) / 2 / max(W, 1); cy = (b[1] + b[3]) / 2 / max(H, 1)
    v = 'top' if cy < 0.34 else 'middle' if cy < 0.67 else 'bottom'
    h = 'left' if cx < 0.34 else 'center' if cx < 0.67 else 'right'
    return f'{v}-{h}'

def serialize(item):
    try:
        d = json.loads(Path(item['ocr_path']).read_text(encoding='utf-8'))
    except Exception:
        d = {}
    W, H = (d.get('image_size') or [1, 1])[:2]
    labels = []
    for ln in (d.get('lines') or [])[:MAX_LINES]:
        t = str(ln.get('text', '')).strip()
        if t and float(ln.get('conf', 0)) >= OCR_MIN_CONF and len(ln.get('bbox', [])) == 4:
            labels.append(f'- "{t}" ({coarse_pos(ln["bbox"], W, H)})')
    parts = []
    if item['caption']:
        parts.append(f"Diagram caption: {item['caption']}")
    parts.append('Text labels in the diagram (with position):\n' + '\n'.join(labels)
                 if labels else '(No reliable text labels were detected in the diagram.)')
    return '\n'.join(parts)

rows = load_test(N_EVAL)
print(f'test-вопросов: {len(rows)}\n--- пример сериализации ---')
print(serialize(rows[0])[:300])

test-вопросов: 3088
--- пример сериализации ---
Diagram caption: A diagram showing the different lobes of the lungs.
Text labels in the diagram (with position):
- "Lung lobes" (top-center)
- "ae CL) Superior Lobes" (top-center)
- "(_] Middte Lobe" (top-center)
- "(J Lobes" (middle-center)
- "oF" (middle-center)
- "Cardiac" (middle-right)
- "notch


---
## Промпт + вызов LLM + парсинг ответа

In [ ]:
#  4. Промпт, вызов, парсинг буквы 
def build_prompt(item):
    opts = '\n'.join(f'{LETTERS[i]}) {o}' for i, o in enumerate(item['options']))
    return (
        'You are answering a multiple-choice question about a science diagram.\n'
        'You cannot see the image, but the diagram has been parsed into the structure below.\n'
        'Use the labels, their positions, and the caption to reason about the answer.\n\n'
        f'{serialize(item)}\n\n'
        f"Question: {item['question']}\n"
        f'Options:\n{opts}\n\n'
        f"Answer with ONLY the letter ({'/'.join(LETTERS[:len(item['options'])])}) of the best option."
    )

def parse_letter(text, n_opts):
    if not text:
        return None
    m = re.search(r'\b([A-H])\b', text.strip().upper())
    if m and LETTERS.index(m.group(1)) < n_opts:
        return LETTERS.index(m.group(1))
    return None

def answer(model, item):
    try:
        r = client.chat.completions.create(model=model, temperature=0, max_tokens=MAX_TOK,
                messages=[{'role': 'user', 'content': build_prompt(item)}])
        txt = r.choices[0].message.content
        idx = parse_letter(txt, len(item['options']))
        if idx is None:                              # fallback: текст варианта в ответе
            low = (txt or '').lower()
            for i, o in enumerate(item['options']):
                if o.lower() in low:
                    idx = i; break
        return idx
    except Exception:
        return None

print('smoke pred:', answer(MODELS[0], rows[0]), '| gold:', rows[0]['correct'])

smoke pred: None | gold: 1


---
## Оценка: accuracy на test + сравнение моделей

Каждый вопрос = 1 API-вызов на модель. `parsed` — сколько ответов удалось распарсить
(непарс засчитывается неверным, т.е. accuracy — нижняя оценка).

In [5]:
# ── 5. Прогон по моделям + таблица ────────────────────────────────────────────
import pandas as pd

per_model = {}
for model in MODELS:
    t0 = time.time(); correct = parsed = done = 0
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = {ex.submit(answer, model, r): i for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            i = futs[fut]; idx = fut.result()
            if idx is not None:
                parsed += 1
            if idx == rows[i]['correct']:
                correct += 1
            done += 1
            if done % 250 == 0:                      # прогресс на длинном прогоне
                print(f"  [{model}] {done}/{len(rows)} | acc={correct/done:.3f} "
                      f"parsed={parsed}/{done} | {time.time()-t0:.0f}s")
    per_model[model] = {'accuracy': round(correct / max(len(rows), 1), 4),
                        'parsed_ok': parsed, 'n_eval': len(rows), 'seconds': round(time.time() - t0, 1)}
    print(f"{model}: acc={per_model[model]['accuracy']} parsed={parsed}/{len(rows)} "
          f"({per_model[model]['seconds']}s)")

tbl = [('Random baseline', 0.25, '—')] + \
      [(m, per_model[m]['accuracy'], f"{per_model[m]['parsed_ok']}/{len(rows)}") for m in MODELS]
df = pd.DataFrame(tbl, columns=['Модель', 'Accuracy', 'Parsed']).sort_values('Accuracy', ascending=False)
print(f'\nAI2D test, N={len(rows)}\n')
print(df.to_string(index=False))

  [openai/gpt-oss-20b] 250/3088 | acc=0.668 parsed=205/250 | 392s
  [openai/gpt-oss-20b] 500/3088 | acc=0.658 parsed=413/500 | 868s
  [openai/gpt-oss-20b] 750/3088 | acc=0.623 parsed=605/750 | 1545s
  [openai/gpt-oss-20b] 1000/3088 | acc=0.628 parsed=802/1000 | 2234s
  [openai/gpt-oss-20b] 1250/3088 | acc=0.623 parsed=997/1250 | 2848s
  [openai/gpt-oss-20b] 1500/3088 | acc=0.618 parsed=1190/1500 | 3504s
  [openai/gpt-oss-20b] 1750/3088 | acc=0.614 parsed=1390/1750 | 3886s
  [openai/gpt-oss-20b] 2000/3088 | acc=0.612 parsed=1578/2000 | 4327s
  [openai/gpt-oss-20b] 2250/3088 | acc=0.613 parsed=1783/2250 | 4723s
  [openai/gpt-oss-20b] 2500/3088 | acc=0.611 parsed=1979/2500 | 5252s
  [openai/gpt-oss-20b] 2750/3088 | acc=0.612 parsed=2175/2750 | 5634s
  [openai/gpt-oss-20b] 3000/3088 | acc=0.616 parsed=2376/3000 | 6080s
openai/gpt-oss-20b: acc=0.6153 parsed=2446/3088 (6320.6s)
  [deepseek/deepseek-v4-pro] 250/3088 | acc=0.628 parsed=168/250 | 259s
  [deepseek/deepseek-v4-pro] 500/3088 | acc

In [ ]:
#  Сохранение метрик 
metrics = {'dataset': 'AI2D', 'task': 'MCQ',
           'approach': 'LLM-reasoner over parsed diagram (caption + OCR + positions); non-VLM',
           'split': 'test', 'n_eval': len(rows), 'random_baseline': 0.25,
           'reference': {'CoG-DQA': 0.79, 'Qwen-VLM': 0.73, 'GraphColBERT-FiLM': 0.348},
           'per_model': per_model}
(OUTPUT_DIR / 'metrics_all.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))

{
  "dataset": "AI2D",
  "task": "MCQ",
  "approach": "LLM-reasoner over parsed diagram (caption + OCR + positions); non-VLM",
  "split": "test",
  "n_eval": 3088,
  "random_baseline": 0.25,
  "reference": {
    "CoG-DQA": 0.79,
    "Qwen-VLM": 0.73,
    "GraphColBERT-FiLM": 0.348
  },
  "per_model": {
    "openai/gpt-oss-20b": {
      "accuracy": 0.6153,
      "parsed_ok": 2446,
      "n_eval": 3088,
      "seconds": 6320.6
    },
    "deepseek/deepseek-v4-pro": {
      "accuracy": 0.1852,
      "parsed_ok": 615,
      "n_eval": 3088,
      "seconds": 1030.6
    }
  }
}
